## **Embeddings Generation using OpenAI SDK**

In [1]:
from openai import OpenAI
client = OpenAI()

response = client.embeddings.create(
    input="Your text string goes here",
    model="text-embedding-3-small"
)
# Note that this is not token level embedding, but sentence level embedding
print(response.data[0].embedding)

[0.005130767822265625, 0.0171966552734375, -0.0187225341796875, -0.0185546875, -0.047271728515625, -0.030303955078125, 0.027679443359375, 0.0036640167236328125, 0.01123809814453125, 0.00641632080078125, -0.00168609619140625, 0.0158233642578125, -0.0012884140014648438, -0.0078125, 0.05987548828125, 0.050262451171875, -0.0274658203125, 0.009918212890625, -0.04034423828125, 0.049957275390625, -0.0004527568817138672, 0.03021240234375, -0.01372528076171875, 0.03289794921875, 0.017303466796875, 0.01678466796875, -0.0017557144165039062, 0.020416259765625, 0.040771484375, -0.037750244140625, -0.026092529296875, -0.04998779296875, 0.024139404296875, -0.05517578125, -0.0322265625, 0.042388916015625, 0.064697265625, 0.01471710205078125, -0.0156402587890625, -0.04132080078125, 0.02215576171875, 0.007335662841796875, 0.044891357421875, 0.007068634033203125, -0.0240936279296875, 0.052398681640625, -0.02008056640625, -0.03216552734375, 0.0162200927734375, 0.046478271484375, 0.0238494873046875, -0.018

## Notes: Text Embeddings

### What Are Embeddings?

An **embedding** is a dense numerical vector that captures the semantic meaning of a piece of text. Instead of treating words as discrete symbols, embeddings map text into a continuous vector space where **semantically similar texts are geometrically close** (measured by cosine similarity or dot product).

### Sentence-Level vs Token-Level Embeddings

| Type | What it captures | Typical use |
|---|---|---|
| **Token-level** | One vector per token (word/subword). Used internally by transformers in each layer. | Attention mechanisms, NER, POS tagging |
| **Sentence-level** (this notebook) | One vector for the entire input string, pooled from token representations. | Semantic search, RAG retrieval, clustering, classification |

OpenAI's embedding endpoint returns a **single vector per input string** — this is sentence-level embedding. Internally the model still computes token-level representations, but the API returns a pooled aggregate.

### The `text-embedding-3-small` Model

- **Dimensions**: 1536 (default). Can be reduced via the `dimensions` parameter to trade accuracy for storage/speed.
- **Max input tokens**: 8191 tokens.
- **Normalization**: Vectors are L2-normalized (unit length), so cosine similarity = dot product.
- **Cost-effective**: Cheaper and faster than `text-embedding-3-large` (3072 dims), suitable for most retrieval tasks.

### How Embeddings Are Generated (Simplified)

```
Input text
    ↓
Tokenizer (BPE) → token IDs
    ↓
Transformer encoder (multiple layers of self-attention + FFN)
    ↓
Token-level hidden states (one vector per token)
    ↓
Pooling (e.g., mean pooling or [CLS] token)
    ↓
Single embedding vector (1536 dims)
```

The model is trained with a **contrastive objective** — pairs of related texts are pushed closer together while unrelated pairs are pushed apart in vector space.

### Common Applications

1. **Semantic Search / RAG** — embed documents and queries, retrieve by nearest-neighbor similarity.
2. **Clustering** — group similar documents using K-means or HDBSCAN on embedding vectors.
3. **Classification** — use embeddings as feature vectors for downstream classifiers.
4. **Anomaly Detection** — identify outlier documents that are far from any cluster centroid.
5. **Recommendation** — find items with similar embeddings to a user's history.

### Key Considerations

- **Garbage in, garbage out**: embedding quality depends on input quality. Preprocessing (cleaning, chunking) matters.
- **Chunking strategy**: for long documents, splitting into smaller chunks (100–500 tokens) before embedding improves retrieval precision — a single embedding cannot capture nuance across thousands of tokens.
- **Distance metric**: since OpenAI embeddings are normalized, cosine similarity and dot product are equivalent. Euclidean distance also works but is less common.

### Sources
- [OpenAI Embeddings Guide](https://platform.openai.com/docs/guides/embeddings)
- [OpenAI Embedding Models](https://platform.openai.com/docs/models/embeddings)